# Ep. 03 — Memoria de conversación, ventana de contexto y ventana deslizante

Notebook técnico: cuatro progresiones. Cada request es independiente salvo que **tú** envíes el historial.


## Setup

Shared helper: real API if `OPENAI_API_KEY` is set, otherwise offline mock (including context length exceeded).


In [ ]:
import os
import json
from typing import Any

# --- Vanilla chat helper: real OpenAI API if OPENAI_API_KEY is set, else offline mock ---
API_KEY = os.environ.get("OPENAI_API_KEY", "").strip()
MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
USE_REAL = bool(API_KEY)

# Simulated tiny context window for demos when mocking (characters ≈ coarse "tokens")
MOCK_CONTEXT_LIMIT_CHARS = 800


def _approx_tokens(messages: list[dict]) -> int:
    return max(1, sum(len(m.get("content", "")) for m in messages) // 4)


def chat(messages: list[dict], *, max_tokens: int = 200) -> str:
    """Send chat messages. Raises RuntimeError with 'context length exceeded' when over budget."""
    if USE_REAL:
        import urllib.request
        body = json.dumps({
            "model": MODEL,
            "messages": messages,
            "max_tokens": max_tokens,
            "temperature": 0,
        }).encode()
        req = urllib.request.Request(
            "https://api.openai.com/v1/chat/completions",
            data=body,
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
            },
            method="POST",
        )
        try:
            with urllib.request.urlopen(req, timeout=60) as resp:
                data = json.loads(resp.read().decode())
            return data["choices"][0]["message"]["content"]
        except Exception as e:
            text = str(e)
            # Surface provider context errors clearly
            if "context_length" in text.lower() or "maximum context" in text.lower() or "context length" in text.lower():
                raise RuntimeError("context length exceeded: " + text) from e
            # Try to read HTTP error body
            if hasattr(e, "read"):
                try:
                    err_body = e.read().decode()
                    if "context" in err_body.lower() and "length" in err_body.lower():
                        raise RuntimeError("context length exceeded: " + err_body) from e
                except RuntimeError:
                    raise
                except Exception:
                    pass
            raise
    # Offline mock
    total_chars = sum(len(m.get("content", "")) for m in messages)
    if total_chars > MOCK_CONTEXT_LIMIT_CHARS:
        raise RuntimeError(
            f"context length exceeded: mock limit {MOCK_CONTEXT_LIMIT_CHARS} chars, got {total_chars}"
        )
    # Find last user message and any earlier user facts
    users = [m["content"] for m in messages if m.get("role") == "user"]
    if not users:
        return "(no user message)"
    last = users[-1]
    # If history includes a prior fact about a secret/codeword, recall it
    history_text = "\n".join(users[:-1])
    if "lembra" in last.lower() or "remember" in last.lower() or "recuerd" in last.lower() or "qual" in last.lower() or "what" in last.lower() or "cuál" in last.lower() or "cual" in last.lower():
        # Look for CODEWORD / palavra / palabra patterns in earlier turns
        for u in users[:-1]:
            for marker in ("CODEWORD:", "palavra-chave:", "palabra clave:", "secret:", "segredo:", "secreto:"):
                if marker.lower() in u.lower():
                    # extract after marker
                    idx = u.lower().index(marker.lower())
                    val = u[idx + len(marker):].strip().split()[0]
                    return f"Recalled from history: {val}"
        if len(users) == 1:
            return "I have no prior messages in this request, so I don't know."
        return "I don't see that fact in the provided history."
    if len(users) == 1:
        return f"Got it. (stateless ack) Stored nothing server-side."
    return f"OK. History turns={len(users)}. Last: {last[:80]}"


def sliding_window(messages: list[dict], max_messages: int) -> list[dict]:
    """Keep system (if any) + the most recent messages up to max_messages total dialog slots."""
    if not messages:
        return []
    system = [m for m in messages if m.get("role") == "system"]
    dialog = [m for m in messages if m.get("role") != "system"]
    keep_dialog = max(0, max_messages - len(system))
    trimmed = dialog[-keep_dialog:] if keep_dialog else []
    return system + trimmed


print("USE_REAL_API =", USE_REAL, "| MODEL =", MODEL if USE_REAL else "(mock)")
print("MOCK_CONTEXT_LIMIT_CHARS =", MOCK_CONTEXT_LIMIT_CHARS)


## 1) El LLM olvida

Dos llamadas **separadas**. La segunda no lleva historial, así que el modelo no recuerda el hecho de la primera.


In [ ]:
hecho = "CODEWORD: ORBIT-77"
r1 = chat([{"role": "user", "content": f"Recuerda este secreto: {hecho}"}])
print("Llamada 1:", r1)

r2 = chat([{"role": "user", "content": "¿Cuál es el CODEWORD que te dije?"}])
print("Llamada 2 (sin historial):", r2)
print("→ Esperado: NO sabe ORBIT-77.")


## 2) Armar el historial

Acumula mensajes en una lista (antiguo → reciente). Reenvía la lista en cada turno para que el modelo "recuerde".


In [ ]:
historial: list[dict] = []
historial.append({"role": "user", "content": "Recuerda este secreto: CODEWORD: ORBIT-77"})
a1 = chat(historial)
historial.append({"role": "assistant", "content": a1})
print("Tras turno 1, len =", len(historial))

historial.append({"role": "user", "content": "¿Cuál es el CODEWORD que te dije?"})
a2 = chat(historial)
historial.append({"role": "assistant", "content": a2})
print("Con historial:", a2)
print("→ Esperado: recupera ORBIT-77 de la lista que enviaste.")


## 3) Romper la ventana de contexto

Sigue añadiendo hasta que el proveedor devuelva **context length exceeded** (error real si hay clave; si no, mock fiel).


In [ ]:
big = list(historial)
blob = "PADDING " * 50
errored = False
for i in range(1, 40):
    big.append({"role": "user", "content": f"turno-{i}: {blob}"})
    try:
        _ = chat(big, max_tokens=16)
        print(f"ok crecimiento {i}, messages={len(big)}")
    except RuntimeError as e:
        print("ERROR:", e)
        errored = True
        break
print("Recibió context length exceeded =", errored)


## 4) Ventana deslizante

Cuando el presupuesto se llena, elimina los mensajes de diálogo **más antiguos** (conserva `system`). Olvida el inicio, mantiene el hilo reciente — y deja de fallar.


In [ ]:
msgs = [{"role": "system", "content": "Eres un asistente conciso."}]
msgs.append({"role": "user", "content": "Hecho inicial CODEWORD: ORBIT-77 — solo al inicio."})
msgs.append({"role": "assistant", "content": "Ok, hecho inicial."})
for i in range(20):
    msgs.append({"role": "user", "content": f"Tema reciente {i}: " + ("x" * 40)})
    msgs.append({"role": "assistant", "content": f"Ack {i}"})

print("Antes del trim: messages =", len(msgs), "approx_tokens ≈", _approx_tokens(msgs))
try:
    chat(msgs, max_tokens=16)
    print("Inesperado: la llamada grande pasó")
except RuntimeError as e:
    print("Error antes del trim (esperado):", e)

trimmed = sliding_window(msgs, max_messages=8)
print("Tras sliding_window(max_messages=8):", len(trimmed), "messages")
for m in trimmed:
    print(" ", m["role"], ":", m["content"][:60].replace("\n", " "), "...")

out = chat(trimmed + [{"role": "user", "content": "¿Cuál era el CODEWORD del principio?"}])
print("Pregunta sobre el inicio tras trim:", out)
print("→ El CODEWORD inicial suele desaparecer; el hilo reciente aún cabe.")
out2 = chat(trimmed + [{"role": "user", "content": "¿Cuál fue el último índice de tema reciente?"}])
print("Pregunta sobre lo reciente:", out2)
